## Example 1 — version and FFmpeg major (no GPU)

In [ ]:
import importlib.metadata
import torchcodec

assert importlib.metadata.version("torchcodec") == torchcodec.__version__, torchcodec.__version__
print("torchcodec", torchcodec.__version__, "on FFmpeg major", torchcodec.ffmpeg_major_version)

## Example 2 — audio, vLLM's call: four formats decoded and resampled to 16 kHz (no GPU)

In [ ]:
import math
import torch
from torchcodec.decoders import AudioDecoder, WavDecoder
from torchcodec.encoders import AudioEncoder

rate = 44100
tone = (0.5 * torch.sin(2 * math.pi * 440 * torch.arange(rate) / rate)).unsqueeze(0)      # one second, one channel, 440 Hz
report = []
# format, the codec FFmpeg picks for it, the rate to encode at (Opus only takes its own rates), same-rate error limit
for fmt, codec, encode_rate, limit in (("wav", "pcm_s16le", None, 1e-4), ("flac", "flac", None, 1e-4), ("mp3", "mp3", None, 0.1), ("opus", "opus", 48000, None)):
    encoded = AudioEncoder(tone, sample_rate=rate).to_tensor(format=fmt, sample_rate=encode_rate)
    decoder = AudioDecoder(encoded)
    assert decoder.metadata.codec == codec and decoder.metadata.num_channels == 1, (fmt, decoder.metadata.codec)
    if limit is not None:                                  # decoded at its own rate: the tone comes back sample for sample
        samples = decoder.get_all_samples()
        assert samples.sample_rate == rate and (samples.data[0, :rate] - tone[0]).abs().max() < limit, fmt
    speech = AudioDecoder(encoded, sample_rate=16000).get_all_samples()                    # what vLLM asks for
    peak = torch.fft.rfft(speech.data[0]).abs().argmax().item() * 16000 / speech.data.shape[1]
    assert speech.sample_rate == 16000 and speech.data.shape == (1, 16000) and speech.data.dtype == torch.float32, (fmt, speech.data.shape)
    assert abs(peak - 440) < 2 and 0.4 < speech.data.abs().max() < 0.6, (fmt, peak, speech.data.abs().max())
    report.append(f"{fmt}/{codec} {len(encoded)} bytes")
wav = WavDecoder(AudioEncoder(tone, sample_rate=rate).to_tensor(format="wav")).get_all_samples()   # the FFmpeg-free WAV reader
assert wav.data.shape == (1, rate) and (wav.data - tone).abs().max() < 1e-4
print("decoded and resampled to 16 kHz with the 440 Hz tone intact:", report, "| WavDecoder agrees")

## Example 3 — video on the CPU: encode 30 frames, decode them back, pick frames by index and by time (no GPU)

In [ ]:
import torch
from torchcodec.decoders import VideoDecoder
from torchcodec.encoders import VideoEncoder
from torchcodec.samplers import clips_at_regular_indices
from torchcodec.transforms import Resize

count, height, width = 30, 96, 128
rows, columns = torch.meshgrid(torch.arange(height), torch.arange(width), indexing="ij")
frames = torch.stack([torch.stack([(columns * 2 + 4 * i) % 256, (rows * 2) % 256, torch.full_like(rows, 128)]) for i in range(count)]).to(torch.uint8)
clip = VideoEncoder(frames, frame_rate=30).to_tensor(format="mp4", codec="mpeg4")
decoder = VideoDecoder(clip)
meta = decoder.metadata
assert (len(decoder), meta.codec, meta.width, meta.height, meta.average_fps, meta.num_frames) == (count, "mpeg4", width, height, 30, count), meta
decoded = decoder[:]
assert decoded.shape == frames.shape and decoded.dtype == torch.uint8
# A lossy codec: every frame is close to its source, and closer to its own source frame than to any other (order kept).
distance = (decoded[:, None].float() - frames[None].float()).abs().mean(dim=(2, 3, 4))      # decoded frame x source frame
assert distance.diagonal().max() < 4 and torch.equal(distance.argmin(dim=1), torch.arange(count)), distance.diagonal()
frame = decoder.get_frame_played_at(0.5)
assert abs(frame.pts_seconds - 0.5) < 1e-6 and torch.equal(frame.data, decoded[15])
batch = decoder.get_frames_at([29, 0, 10])
assert torch.equal(batch.data, decoded[[29, 0, 10]]) and batch.pts_seconds.tolist() == [29 / 30, 0.0, 10 / 30]
assert VideoDecoder(clip, dimension_order="NHWC")[3].shape == (height, width, 3)
assert VideoDecoder(clip, transforms=[Resize((48, 64))])[3].shape == (3, 48, 64)
assert clips_at_regular_indices(decoder, num_clips=3, num_frames_per_clip=4).data.shape == (3, 4, 3, height, width)
print(f"{len(clip)} bytes of MPEG-4 -> {count} frames; mean error per frame at most {distance.diagonal().max():.2f} of 255; index, time, layout, resize and clip sampling all work")

## Example 4 — images: PNG/JPEG round trips, WebP and GIF with known pixels (no GPU)

In [ ]:
import base64
import torch
from torchcodec.decoders import decode_gif, decode_image, decode_jpeg, decode_png, decode_webp
from torchcodec.encoders import JpegEncoder, PngEncoder

rows, columns = torch.meshgrid(torch.arange(96), torch.arange(128), indexing="ij")
image = torch.stack([(columns * 2) % 256, (rows * 2) % 256, torch.full_like(rows, 128)]).to(torch.uint8)
png = PngEncoder(image).to_tensor()
assert torch.equal(decode_png(png), image) and torch.equal(decode_image(png), image)        # lossless: exact
jpeg = JpegEncoder(image).to_tensor(quality=95)
assert decode_jpeg(jpeg).shape == image.shape and (decode_jpeg(jpeg).float() - image.float()).abs().mean() < 2
assert decode_jpeg(jpeg, mode="GRAY").shape == (1, 96, 128)
# One 4x3 picture with known pixels, stored twice in 64 bytes: as a lossless WebP and as a GIF.
webp = base64.b64decode("UklGRjgAAABXRUJQVlA4TCwAAAAvA4AAACcgECBI8R9pQyCQpLA/7AIBgjT/UeY/8MmxVSAQSEMOkQxxRP9Deg==")
gif = base64.b64decode("R0lGODdhBAADAIIAAP///wD/ABAgQP8AAAAA/wAAAAAAAAAAACwAAAAABAADAAAIDQAHBCAAQCBBAQgFBAQAOw==")
colours = [[255, 0, 0], [0, 255, 0], [0, 0, 255], [255, 255, 255]]
expected = torch.tensor([colours, colours, [[16, 32, 64]] * 4], dtype=torch.uint8).permute(2, 0, 1)
assert torch.equal(decode_webp(webp), expected) and torch.equal(decode_image(webp), expected)
assert torch.equal(decode_gif(gif), expected) and torch.equal(decode_image(gif), expected)
assert decode_gif(gif, mode="RGB_ALPHA").shape == (4, 3, 4)
try:
    decode_png(jpeg)
    raise AssertionError("a JPEG was accepted as a PNG")
except RuntimeError as error:
    print("PNG exact; JPEG close; WebP and GIF give the known pixels; a JPEG handed to the PNG decoder is refused:", type(error).__name__)

## Example 5 — on the GPU: six clips decoded on both of torchcodec's GPU back ends ("ffmpeg", which vLLM uses, must be right on all; the default "nvdec" may miss only the two known upstream cases) and a JPEG decoded by nvJPEG, each compared with the CPU result

In [ ]:
import torch
from torchcodec.decoders import VideoDecoder, decode_jpeg, set_cuda_backend
from torchcodec.encoders import JpegEncoder, VideoEncoder

assert torch.cuda.is_available(), "requires-gpu cell ran but torch sees no CUDA device"
def pictures(height, step):            # 30 frames, 320 wide: stripes that move, and a blue level of step x frame number
    rows, columns = torch.meshgrid(torch.arange(height), torch.arange(320), indexing="ij")
    return torch.stack([torch.stack([(columns * 2 + 4 * i) % 256, (rows * 2) % 256, torch.full_like(rows, 128 + step * i)]) for i in range(30)]).to(torch.uint8)
# One video after another, the way a server meets them: four codecs, then a second HEVC clip of another height, then an
# H.264 clip whose frames differ only by the moving stripes.
cases = (("H.264", "h264_nvenc", 240, 4), ("HEVC", "hevc_nvenc", 240, 4), ("AV1", "av1_nvenc", 240, 4), ("MPEG-4", "mpeg4", 240, 4),
         ("HEVC again, another height", "hevc_nvenc", 256, 4), ("H.264, stripes only", "h264_nvenc", 240, 0))
KNOWN_UPSTREAM_DEFECTS = {"MPEG-4", "H.264, stripes only"}
gaps = {"ffmpeg": {}, "nvdec": {}}
for label, codec, height, step in cases:
    # Only GPUs from the RTX 40 generation on (compute capability 8.9) can encode AV1; older GPUs cannot, and that is not a defect.
    if codec == "av1_nvenc" and torch.cuda.get_device_capability() < (8, 9):
        print(f"SKIP: compute capability {torch.cuda.get_device_capability()} has no AV1 encoder -> the AV1 clip is not made or decoded"); continue
    clip = VideoEncoder(pictures(height, step), frame_rate=30).to_tensor(format="mp4", codec=codec)
    on_cpu = VideoDecoder(clip)[:].int()
    for backend in gaps:
        with set_cuda_backend(backend):
            decoder = VideoDecoder(clip, device="cuda")
            on_gpu = decoder[:]
        # `cpu_fallback` is how torchcodec says "the GPU's decoder refused this stream and the CPU did the work".
        assert on_gpu.is_cuda and not decoder.cpu_fallback, (backend, label, str(decoder.cpu_fallback))
        gap = (on_gpu.cpu().int() - on_cpu).abs()                                          # against the CPU decode of the same clip
        gaps[backend][label] = round(gap.float().mean().item(), 2) if codec == "mpeg4" else gap.max().item()
jpeg = JpegEncoder(pictures(240, 0)[0]).to_tensor(quality=95)
on_gpu, on_cpu = decode_jpeg(jpeg, device="cuda"), decode_jpeg(jpeg)                        # nvJPEG against libjpeg
# The two JPEG decoders smooth sharp colour edges differently, so they are compared on the average, not sample by sample.
jpeg_gap = (on_gpu.cpu().float() - on_cpu.float()).abs().mean().item()
assert on_gpu.is_cuda and on_gpu.shape == on_cpu.shape and jpeg_gap < 2, jpeg_gap
print(f"nvJPEG decodes on {on_gpu.device}, {jpeg_gap:.2f} of 255 from libjpeg on average; largest GPU-to-CPU difference per clip (MPEG-4: the average): {gaps}")
# Upstream's own tolerance between its GPU and CPU decodes is 3 of 255 on every sample; MPEG-4 is judged on its average, 2.
wrong = {backend: {label: gap for label, gap in per_clip.items() if gap > (2 if label == "MPEG-4" else 3)} for backend, per_clip in gaps.items()}
assert not wrong["ffmpeg"], f"the 'ffmpeg' GPU back end (the one vLLM uses) returned other pictures than the CPU decode for {len(wrong['ffmpeg'])} of {len(gaps['ffmpeg'])} clips: {wrong['ffmpeg']}"
for label in sorted(KNOWN_UPSTREAM_DEFECTS & wrong["nvdec"].keys()):
    print(f"KNOWN UPSTREAM DEFECT: the default 'nvdec' back end decodes the clip '{label}' wrongly ({wrong['nvdec'][label]} of 255)")
for label in sorted(KNOWN_UPSTREAM_DEFECTS & gaps["nvdec"].keys() - wrong["nvdec"].keys()):
    print(f"NOTE: the clip '{label}' now decodes right on the default 'nvdec' back end: upstream may have fixed it; drop it from KNOWN_UPSTREAM_DEFECTS")
new = {label: gap for label, gap in wrong["nvdec"].items() if label not in KNOWN_UPSTREAM_DEFECTS}
assert not new, f"the default 'nvdec' GPU back end returned other pictures than the CPU decode for a clip that is NOT one of the two known upstream defects: {new}"